# TokenShip LoRA Training Notebook
### Qwen2.5-7B-Instruct | MATH Train Split (~7,500 problems) | Ratios: 0.5, 0.6, 0.7

**Pipeline:**
1. Load MATH train split from HuggingFace
2. Run Qwen inference → save raw CoT (`math_train_cot.jsonl`)
3. LLMLingua-2 compression at ratios 0.5, 0.6, 0.7
4. Train 3 LoRA adapters (one per ratio, 3 epochs each)
5. Download `qwen_lora_ratio_0.5.zip`, `_0.6.zip`, `_0.7.zip`

**Then:** Upload adapters to `tokenship-math500-qwen2.5.ipynb` for eval on MATH-500.

In [1]:
!pip install -q transformers>=4.37.0 accelerate flash-attn --no-build-isolation
print("Done 1")
!pip install -q peft llmlingua sentencepiece
print("Done 2")
!pip install -q datasets protobuf
print("Done 3")

Done 1
Done 2
Done 3


In [2]:
import transformers, torch
print(f"Transformers : {transformers.__version__}")
print(f"PyTorch      : {torch.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")
print(f"GPU          : {torch.cuda.get_device_name(0)}")

Transformers : 5.2.0
PyTorch      : 2.9.0+cu126
CUDA         : True
GPU          : NVIDIA H100 80GB HBM3


In [3]:
import os, re, time, json
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME       = "Qwen/Qwen2.5-7B-Instruct"
MAX_NEW_TOKENS   = 1024
EVAL_BATCH_SIZE  = 64
TRAIN_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 8       # effective batch size = 4 * 8 = 32
TRAIN_EPOCHS     = 3
TARGET_RATIOS    = [0.5, 0.6, 0.7]
WORKING_DIR      = "/kaggle/working"
DEVICE           = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

Device: cuda


In [4]:
from datasets import load_dataset, concatenate_datasets

SUBJECTS = [
    "algebra", "counting_and_probability", "geometry",
    "intermediate_algebra", "number_theory", "prealgebra", "precalculus"
]

all_splits = []
for subj in SUBJECTS:
    ds = load_dataset("EleutherAI/hendrycks_math", subj, split="train")
    print(f"  {subj:30s} → {len(ds)} problems")
    all_splits.append(ds)

math_ds = concatenate_datasets(all_splits)
print(f"\nTotal train problems: {len(math_ds)}")

train_df = pd.DataFrame({
    "Question" : [ex["problem"]  for ex in math_ds],
    "Answer"   : [ex["solution"] for ex in math_ds],
    "Level"    : [ex["level"]    for ex in math_ds],
    "Type"     : [ex["type"]     for ex in math_ds],
}).reset_index(drop=True)

print(f"DataFrame shape: {train_df.shape}")
print(train_df[["Question", "Level", "Type"]].head(3))

README.md: 0.00B [00:00, ?B/s]

algebra/train-00000-of-00001.parquet:   0%|          | 0.00/505k [00:00<?, ?B/s]

algebra/test-00000-of-00001.parquet:   0%|          | 0.00/353k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1744 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1187 [00:00<?, ? examples/s]

...

precalculus/train-00000-of-00001.parquet:   0%|          | 0.00/354k [00:00<?, ?B/s]

precalculus/test-00000-of-00001.parquet:   0%|          | 0.00/242k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/746 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/546 [00:00<?, ? examples/s]

  precalculus                    → 746 problems

Total train problems: 7500
DataFrame shape: (7500, 4)
                                            Question    Level     Type
0  Let \[f(x) = \left\{\n\begin{array}{cl} ax+3, ...  Level 5  Algebra
1  A rectangular band formation is a formation wi...  Level 5  Algebra
2  What is the degree of the polynomial $(4 +5x^3...  Level 3  Algebra


In [5]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, trust_remote_code=True, padding_side="left"
)
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="flash_attention_2",
)
base_model.eval()
print(f"Model on: {next(base_model.parameters()).device}")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model on: cuda:0


In [6]:
def extract_boxed(text):
    str_text = str(text)
    idx = str_text.find("\\boxed{")
    if idx != -1:
        depth, start, end = 1, idx+7, idx+7
        while end < len(str_text) and depth > 0:
            if   str_text[end] == '{': depth += 1
            elif str_text[end] == '}': depth -= 1
            end += 1
        if depth == 0:
            return str_text[start:end-1]
    match = re.search(r"final answer is[:\s]*(.*)", str_text, re.IGNORECASE)
    return match.group(1).strip() if match else str_text.strip()

def normalize_ans(ans):
    ans = str(ans).strip()
    ans = re.sub(r'\s+', ' ', ans)
    ans = re.sub(r'\\,', '', ans)
    return re.sub(r'[,\-]', '', ans).lower()

def is_correct(pred, gt):
    p, g = normalize_ans(extract_boxed(pred)), normalize_ans(extract_boxed(gt))
    if p == g: return True
    try:    return abs(float(p.replace(',','')) - float(g.replace(',',''))) < 1e-6
    except: return False

In [7]:
PROMPTS = {
    "Original": (
        "Solve the following math problem step by step. "
        "Think carefully and show your work. "
        "Put your final answer within \\boxed{{}}."
        "\n\nProblem: {question}"
    ),
}

def make_prompt(question, method="Original"):
    prompt   = PROMPTS[method].format(question=question)
    messages = [{"role": "user", "content": prompt}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [8]:
def evaluate_batched(df, method="Original", max_new_tokens=MAX_NEW_TOKENS,
                     original_avg_tokens=None, model=None):
    _model = model if model is not None else base_model
    all_responses, all_token_counts = [], []
    start_time = time.time()

    prompts_indexed = [(i, make_prompt(row["Question"], method)) for i, row in df.iterrows()]
    prompts_indexed.sort(key=lambda x: len(x[1]))
    sorted_indices = [i for i, _ in prompts_indexed]
    prompts        = [p for _, p in prompts_indexed]

    for batch_start in tqdm(range(0, len(prompts), EVAL_BATCH_SIZE),
                            desc=f"{method} (bs={EVAL_BATCH_SIZE})"):
        batch_prompts = prompts[batch_start : batch_start + EVAL_BATCH_SIZE]
        inputs = tokenizer(
            batch_prompts, return_tensors="pt",
            padding=True, truncation=True, max_length=2048
        ).to(DEVICE)
        input_len = inputs["input_ids"].shape[1]
        with torch.no_grad():
            outputs = _model.generate(
                **inputs, max_new_tokens=max_new_tokens, do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        generated    = outputs[:, input_len:]
        token_counts = (generated != tokenizer.pad_token_id).sum(dim=1).tolist()
        responses    = tokenizer.batch_decode(generated, skip_special_tokens=True)
        all_token_counts.extend(token_counts)
        all_responses.extend(responses)
        del outputs, inputs, generated
        torch.cuda.empty_cache()

    reordered_responses    = [None] * len(df)
    reordered_token_counts = [None] * len(df)
    for sorted_pos, orig_idx in enumerate(sorted_indices):
        reordered_responses[orig_idx]    = all_responses[sorted_pos]
        reordered_token_counts[orig_idx] = all_token_counts[sorted_pos]

    elapsed    = time.time() - start_time
    correct    = sum(is_correct(r, g) for r, g in zip(reordered_responses, df["Answer"]))
    avg_tokens = sum(reordered_token_counts) / len(df)
    return (
        {"Method": method, "Accuracy": round(100*correct/len(df), 2),
         "Avg Tokens": round(avg_tokens, 2), "Latency(s)": round(elapsed/len(df), 3),
         "Act Ratio":  round(avg_tokens/original_avg_tokens, 3) if original_avg_tokens else 1.0,
         "Correct": correct, "Total": len(df)},
        reordered_responses, reordered_token_counts
    )

In [9]:
COT_PATH = f"{WORKING_DIR}/math_train_cot.jsonl"

if os.path.exists(COT_PATH):
    with open(COT_PATH) as f:
        done_ids = {json.loads(line)["id"] for line in f}
    print(f"Resuming: {len(done_ids)}/{len(train_df)} already done")
else:
    done_ids = set()
    print(f"Starting fresh — {len(train_df)} problems to process")

remaining_mask    = ~train_df.index.isin(done_ids)
remaining_df_raw  = train_df[remaining_mask].copy()
remaining_orig_ix = remaining_df_raw.index.tolist()
remaining_df      = remaining_df_raw.reset_index(drop=True)

if len(remaining_df) > 0:
    print(f"Running inference on {len(remaining_df)} problems...")
    _, responses, token_counts = evaluate_batched(remaining_df, method="Original")
    with open(COT_PATH, "a") as f:
        for local_i, (resp, tc) in enumerate(zip(responses, token_counts)):
            orig_idx = remaining_orig_ix[local_i]
            row = train_df.iloc[orig_idx]
            f.write(json.dumps({
                "id"          : int(orig_idx),
                "problem"     : row["Question"],
                "answer"      : row["Answer"],
                "full_cot"    : resp,
                "token_count" : tc,
                "level"       : row["Level"],
                "subject"     : row["Type"],
            }) + "\n")
    print(f"Saved to {COT_PATH}")
else:
    print("All problems already processed — skipping inference.")

with open(COT_PATH) as f:
    cot_records = [json.loads(line) for line in f]

avg_train_tokens = sum(r["token_count"] for r in cot_records) / len(cot_records)
print(f"Total CoT records  : {len(cot_records)}")
print(f"Avg tokens (train) : {avg_train_tokens:.1f}")

Starting fresh — 7500 problems to process
Running inference on 7500 problems...


Original (bs=64):   0%|          | 0/118 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Saved to /kaggle/working/math_train_cot.jsonl
Total CoT records  : 7500
Avg tokens (train) : 593.4


In [10]:
from llmlingua import PromptCompressor

print("Loading LLMLingua-2 on CPU (preserves GPU VRAM for Qwen)...")
llm_lingua = PromptCompressor(
    model_name="microsoft/llmlingua-2-xlm-roberta-large-meetingbank",
    use_llmlingua2=True,
    device_map="cuda",
)
print("LLMLingua-2 ready!")

for ratio in TARGET_RATIOS:
    out_path = f"{WORKING_DIR}/math_train_compressed_ratio{ratio}.jsonl"
    if os.path.exists(out_path):
        with open(out_path) as f:
            n = sum(1 for _ in f)
        print(f"  ratio={ratio} already exists ({n} records), skipping.")
        continue
    print(f"\nCompressing ratio={ratio}  ({len(cot_records)} samples)...")
    t0 = time.time()
    with open(out_path, "w") as f_out:
        for rec in tqdm(cot_records, desc=f"LLMLingua ratio={ratio}"):
            try:
                result     = llm_lingua.compress_prompt(
                    rec["full_cot"], rate=ratio,
                    force_tokens=["\n", ".", "\\boxed"],
                )
                compressed = result["compressed_prompt"]
            except Exception:
                compressed = rec["full_cot"]   # fallback: keep original on error
            f_out.write(json.dumps({
                "id"             : rec["id"],
                "problem"        : rec["problem"],
                "answer"         : rec["answer"],
                "compressed_cot" : compressed,
                "original_tokens": rec["token_count"],
                "level"          : rec.get("level",   ""),
                "subject"        : rec.get("subject", ""),
            }) + "\n")
    print(f"  Done ratio={ratio} in {(time.time()-t0)/60:.1f} min")

print("\nAll compression ratios done!")

Loading LLMLingua-2 on CPU (preserves GPU VRAM for Qwen)...


config.json:   0%|          | 0.00/752 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

...

  Done ratio=0.5 in 3.3 min

Compressing ratio=0.6  (7500 samples)...


LLMLingua ratio=0.6:   0%|          | 0/7500 [00:00<?, ?it/s]

  Done ratio=0.6 in 3.3 min

Compressing ratio=0.7  (7500 samples)...


LLMLingua ratio=0.7:   0%|          | 0/7500 [00:00<?, ?it/s]

  Done ratio=0.7 in 3.3 min

All compression ratios done!


In [11]:
import shutil
from torch.utils.data import Dataset
from peft import LoraConfig, get_peft_model, TaskType
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

class CoTDataset(Dataset):
    """Tokenises (prompt + compressed_cot + EOS) pairs for causal-LM SFT."""
    def __init__(self, records, tokenizer, max_length=1024):
        self.samples = []
        for rec in tqdm(records, desc="Tokenising", leave=False):
            prompt    = make_prompt(rec["problem"], "Original")
            full_text = prompt + rec["compressed_cot"] + tokenizer.eos_token
            enc = tokenizer(
                full_text, truncation=True,
                max_length=max_length, return_tensors="pt"
            )
            self.samples.append({k: v.squeeze(0) for k, v in enc.items()})

    def __len__(self):        return len(self.samples)
    def __getitem__(self, i): return self.samples[i]

In [12]:
# PHASE 3: LoRA Training — one adapter per ratio
# Trains ratio=0.5, then 0.6, then 0.7 sequentially.
# Each adapter is saved + ZIPped immediately after training.
# base_model is cleanly restored between ratios via .base_model.model

for ratio in TARGET_RATIOS:
    compressed_path = f"{WORKING_DIR}/math_train_compressed_ratio{ratio}.jsonl"
    adapter_dir     = f"{WORKING_DIR}/qwen_lora_ratio_{ratio:.1f}"
    zip_path        = f"{WORKING_DIR}/qwen_lora_ratio_{ratio:.1f}.zip"

    if os.path.exists(zip_path):
        print(f"Adapter ratio={ratio} ZIP already exists — skipping.")
        continue

    print(f"\n{'='*65}")
    print(f"  LoRA Training  |  ratio={ratio}  |  epochs={TRAIN_EPOCHS}")
    print(f"{'='*65}")

    with open(compressed_path) as f:
        records = [json.loads(line) for line in f]
    
    print(f"Training samples: {len(records)}")

    # Wrap base_model with a fresh LoRA config
    lora_cfg = LoraConfig(
        r=16, 
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05, 
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )

    lora_model = get_peft_model(base_model, lora_cfg)
    lora_model.print_trainable_parameters()

    dataset        = CoTDataset(records, tokenizer, max_length=1024)
    data_collator  = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    training_args = TrainingArguments(
        output_dir                  = f"{WORKING_DIR}/lora_ckpt_{ratio}",
        num_train_epochs            = TRAIN_EPOCHS,
        per_device_train_batch_size = TRAIN_BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM_STEPS,  # eff. batch = 32
        learning_rate               = 2e-4,
        warmup_ratio                = 0.1,
        lr_scheduler_type           = "cosine",
        bf16                        = True,
        logging_steps               = 10,
        save_strategy               = "no",
        report_to                   = "none",
        dataloader_num_workers      = 2,
    )

    trainer = Trainer(
        model           = lora_model,
        args            = training_args,
        train_dataset   = dataset,
        data_collator   = data_collator,
    )

    trainer.train()

    # Save adapter weights + tokenizer config
    lora_model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)
    print(f"Adapter saved  →  {adapter_dir}")

    # ZIP immediately (don't wait until end — session could timeout)
    shutil.make_archive(f"{WORKING_DIR}/qwen_lora_ratio_{ratio:.1f}", "zip", adapter_dir)
    print(f"ZIP created    →  {zip_path}")

    # Restore clean base_model reference for next ratio
    base_model = lora_model.base_model.model
    
    if hasattr(base_model, 'peft_config'):
        del base_model.peft_config     

    del lora_model, trainer, dataset
    torch.cuda.empty_cache()
    base_model.eval()

    print(f"base_model restored. Ready for next ratio.\n")

print("\n" + "="*65)
print("ALL 3 ADAPTERS TRAINED AND SAVED")
print("="*65)


  LoRA Training  |  ratio=0.5  |  epochs=3
Training samples: 7500
trainable params: 10,092,544 || all params: 7,625,709,056 || trainable%: 0.1323


Tokenising:   0%|          | 0/7500 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
10,1.913621
20,1.681042
30,1.402716
40,1.131238
50,0.963886
60,0.862879
70,0.813631
80,0.774186
90,0.757989
100,0.754137


Adapter saved  →  /kaggle/working/qwen_lora_ratio_0.5
ZIP created    →  /kaggle/working/qwen_lora_ratio_0.5.zip
base_model restored. Ready for next ratio.


  LoRA Training  |  ratio=0.6  |  epochs=3
Training samples: 7500
trainable params: 10,092,544 || all params: 7,625,709,056 || trainable%: 0.1323


...

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(


Tokenising:   0%|          | 0/7500 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
10,1.409385
20,1.235118
30,1.026349
40,0.814150
50,0.686287
60,0.603366
70,0.575544
80,0.544888
90,0.521947
100,0.518950


Adapter saved  →  /kaggle/working/qwen_lora_ratio_0.7
ZIP created    →  /kaggle/working/qwen_lora_ratio_0.7.zip
base_model restored. Ready for next ratio.


ALL 3 ADAPTERS TRAINED AND SAVED


In [13]:
for ratio in TARGET_RATIOS:
    zip_path = f"{WORKING_DIR}/qwen_lora_ratio_{ratio:.1f}.zip"
    if os.path.exists(zip_path):
        size_mb = os.path.getsize(zip_path) / 1e6
        print(f"  OK  qwen_lora_ratio_{ratio:.1f}.zip   ({size_mb:.0f} MB)")
    else:
        print(f"  MISSING  qwen_lora_ratio_{ratio:.1f}.zip")

print()
print("Next step  ->  tokenship-math500-qwen2.5.ipynb")
print("  1. Upload ZIPs as a Kaggle input dataset")
print("  2. PeftModel.from_pretrained(base_model, adapter_dir) for each ratio")
print("  3. evaluate_batched() on MATH-500  (original_avg_tokens=557.91)")
print("  4. Plot accuracy vs token-savings curve  (LoRA vs Truncation)")

  OK  qwen_lora_ratio_0.5.zip   (40 MB)
  OK  qwen_lora_ratio_0.6.zip   (40 MB)
  OK  qwen_lora_ratio_0.7.zip   (40 MB)

Next step  ->  tokenship-math500-qwen2.5.ipynb
  1. Upload ZIPs as a Kaggle input dataset
  2. PeftModel.from_pretrained(base_model, adapter_dir) for each ratio
  3. evaluate_batched() on MATH-500  (original_avg_tokens=557.91)
  4. Plot accuracy vs token-savings curve  (LoRA vs Truncation)


In [16]:
print('Fun')

Fun


In [17]:
from IPython.display import FileLink

FileLink('/kaggle/working/qwen_lora_ratio_0.5.zip')

/kaggle/working/qwen_lora_ratio_0.5.zip

In [ ]:
!ls -lh /kaggle/working